# Customer Churn · Telco

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nicolenki7/Customer-Churn-Project/blob/main/Customer_Churn_Python%2BSQlite.ipynb)

## La pregunta de negocio

Una operadora pierde **uno de cada cuatro clientes**. Captar uno nuevo cuesta entre 5 y 7 veces más que retener al que ya está, así que la pregunta que importa no es *cuánta* gente se va, sino **quién se va a ir, cuándo y por qué**.

Este cuaderno responde tres cosas, en ese orden:

1. **Cuánto** pesa la baja y cómo se reparte por segmento.
2. **Qué** variables empujan la decisión (modelo logístico, interpretable).
3. **Dónde** conviene actuar: los segmentos con más riesgo y más volumen.

**Datos.** [IBM Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d): 7.043 clientes y 21 columnas, una fila por cliente. El fichero se descarga solo y se verifica por `sha256`, así que cualquiera obtiene exactamente estos números.


In [ ]:
import hashlib, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path('data/raw/Telco-Customer-Churn.csv')
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
SHA256 = '16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91'

if not RAW.exists():
    RAW.parent.mkdir(parents=True, exist_ok=True)
    print('Descargando el dataset de IBM...')
    urllib.request.urlretrieve(URL, RAW)

obtenido = hashlib.sha256(RAW.read_bytes()).hexdigest()
assert obtenido == SHA256, 'El fichero no coincide con el checksum esperado'
print(f'Datos OK: {RAW.stat().st_size / 1024:.0f} KB · sha256 {obtenido[:12]}...')


## 1. Carga y limpieza

Tres decisiones que conviene justificar antes de tocar nada:

- **`TotalCharges` llega como texto.** Al convertirlo aparecen 11 valores vacíos, y los 11 son clientes con `tenure = 0`: altas del mes que todavía no tienen factura acumulada. No es un dato corrupto, así que se imputa a 0 en lugar de tirar la fila (perderíamos a los clientes más nuevos, justo los que más se van).
- **`No internet service` es un `No` disfrazado** en los seis servicios de valor añadido. Se unifica para no partir la misma categoría en dos columnas distintas.
- **`customerID` es un identificador, no una variable.** Se elimina antes de modelar.


In [ ]:
df = pd.read_csv(RAW)
print(f'{df.shape[0]} filas x {df.shape[1]} columnas')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
vacios = df['TotalCharges'].isna()
print(f'TotalCharges no numéricos: {int(vacios.sum())}')
print(f'Todos con tenure=0: {bool((df.loc[vacios, chr(116) + 101 + 110 + 117 + 114 + 101] == 0).all())}')

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
             'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in servicios:
    df[col] = df[col].replace('No internet service', 'No')

df['TotalCharges'] = df['TotalCharges'].fillna(0.0)
df = df.drop(columns=['customerID'])

bajas = int(df['Churn'].sum())
print(f'Churn global: {bajas / len(df):.1%} ({bajas} de {len(df)})')


## 2. Feature engineering en SQL

Los datos ya limpios se cargan en una base SQLite en memoria y se enriquecen con SQL. Es la parte que le da sentido al nombre del cuaderno: dos segmentos (`Seniority_Category` y `Monthly_Charges_Level`) que después explican casi todo el riesgo, calculados con `CASE WHEN` en lugar de con veinte líneas de `apply`.

El resultado también se exporta como CSV para quien quiera seguir el análisis en Tableau o Power BI.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
df.to_sql('churn_data', conn, if_exists='replace', index=False)

sql = '''
SELECT *,
  CASE WHEN tenure <= 12 THEN '0-1 año'
       WHEN tenure <= 36 THEN '1-3 años'
       WHEN tenure <= 60 THEN '3-5 años'
       ELSE '5+ años' END AS Seniority_Category,
  CASE WHEN MonthlyCharges < 30 THEN 'Bajo'
       WHEN MonthlyCharges < 80 THEN 'Medio'
       ELSE 'Alto' END AS Monthly_Charges_Level
FROM churn_data;
'''

df_fe = pd.read_sql_query(sql, conn)
conn.close()
print(f'{df_fe.shape[1]} columnas tras el SQL')

Path('outputs').mkdir(exist_ok=True)
df_fe.to_csv('outputs/telco_churn_feature_engineered_for_tableau.csv', index=False)


## 3. La baja por segmento

Antes de modelar conviene mirar. Los tres gráficos de abajo se guardan como SVG en `docs/img/` y son los mismos que ilustran el README, así que lo que se ve aquí y lo que se ve en el repo no puede divergir.


In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({'svg.fonttype': 'none', 'font.family': 'DejaVu Sans',
                     'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.facecolor': 'white'})
AZUL, ROJO, GRIS = '#0f62fe', '#da1e28', '#8d8d8d'
IMG = Path('docs/img')
IMG.mkdir(parents=True, exist_ok=True)

def guardar(fig, nombre):
    fig.tight_layout()
    fig.savefig(IMG / nombre, format='svg', bbox_inches='tight')
    plt.close(fig)
    print('guardado docs/img/' + nombre)

fig, ax = plt.subplots(figsize=(7, 1.5))
ax.barh([0], [1 - tasa], color=GRIS, height=.5)
ax.barh([0], [tasa], left=[1 - tasa], color=ROJO, height=.5)
ax.set_xlim(0, 1); ax.set_ylim(-.6, .6); ax.axis('off')
ax.text((1 - tasa) / 2, 0, f'se queda {1 - tasa:.1%}', ha='center', va='center',
        color='white', fontweight='bold')
ax.text(1 - tasa / 2, 0, f'churn {tasa:.1%}', ha='center', va='center',
        color='white', fontweight='bold')
ax.set_title(f'{len(df):,} clientes: {bajas} se dieron de baja', loc='left')
guardar(fig, 'churn-tasa.svg')


In [ ]:
def tasa_por(col, orden=None):
    s = df_fe.groupby(col)['Churn'].agg(['mean', 'size'])
    return s.loc[orden] if orden else s.sort_values('mean', ascending=False)

por_contrato = tasa_por('Contract', ['Month-to-month', 'One year', 'Two year'])
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(por_contrato.index, por_contrato['mean'] * 100, color=[ROJO, AZUL, AZUL])
for i, (m, n) in enumerate(zip(por_contrato['mean'], por_contrato['size'])):
    ax.text(i, m * 100 + 1, f'{m:.1%}', ha='center', fontweight='bold')
    ax.text(i, -3.2, f'n={n}', ha='center', fontsize=9, color=GRIS)
ax.set_ylabel('churn (%)'); ax.set_ylim(0, 48)
ax.set_title('El contrato mensual concentra el riesgo', loc='left')
ax.tick_params(axis='x', length=0, pad=14)
guardar(fig, 'churn-contrato.svg')

antig = tasa_por('Seniority_Category', ['0-1 año', '1-3 años', '3-5 años', '5+ años'])
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(antig.index, antig['mean'] * 100, color=[ROJO, ROJO, AZUL, AZUL])
for i, m in enumerate(antig['mean']):
    ax.text(i, m * 100 + 1, f'{m:.1%}', ha='center', fontweight='bold')
ax.set_ylabel('churn (%)'); ax.set_ylim(0, 58)
ax.set_title('El primer año es la ventana crítica', loc='left')
ax.tick_params(axis='x', length=0)
guardar(fig, 'churn-antiguedad.svg')

print((por_contrato['mean'] * 100).round(1).to_string())
print()
print((antig['mean'] * 100).round(1).to_string())


## 4. Qué empuja la baja

Regresión logística con `class_weight='balanced'`: la base está desbalanceada (26.5 % de bajas), así que sin ese ajuste el modelo aprendería a decir 'se queda' y acertaría el 73 % de las veces sin haber aprendido nada.

Las categóricas van a *one-hot* con `drop='first'` (evita colinealidad) y las numéricas se estandarizan, para que los coeficientes sean comparables entre sí. Se reporta **AUC** y no solo *accuracy*: con clases desbalanceadas el accuracy esconde los errores que importan.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = df_fe.drop(columns=['Churn', 'Seniority_Category', 'Monthly_Charges_Level'])
y = df_fe['Churn']
cat = X.select_dtypes(exclude='number').columns.tolist()
num = [c for c in X.columns if c not in cat]

modelo = Pipeline([
    ('pre', ColumnTransformer([
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat),
        ('num', StandardScaler(), num)])),
    ('lr', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
modelo.fit(X_tr, y_tr)
prob = modelo.predict_proba(X_te)[:, 1]
print(f'AUC {roc_auc_score(y_te, prob):.3f} · accuracy {accuracy_score(y_te, prob > .5):.3f}')
print(f'train {len(X_tr)} / test {len(X_te)}')


In [ ]:
coefs = pd.DataFrame({
    'feature': modelo.named_steps['pre'].get_feature_names_out(),
    'coef': modelo.named_steps['lr'].coef_[0],
})
coefs['odds_ratio'] = np.exp(coefs['coef'])
coefs = coefs.reindex(coefs['coef'].abs().sort_values(ascending=False).index)
coefs.to_csv('outputs/churn_feature_importance.csv', index=False)

top = coefs.head(8)[::-1]
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(range(len(top)), top['coef'], color=[ROJO if c > 0 else AZUL for c in top['coef']])
ax.set_yticks(range(len(top)))
ax.set_yticklabels([f.replace('cat__', '').replace('num__', '').replace('_', ' ')
                    for f in top['feature']], fontsize=9.5)
ax.axvline(0, color='#333', linewidth=.8)
ax.set_xlabel('coeficiente logístico (positivo = más churn)')
ax.set_title('Qué empuja la baja', loc='left')
guardar(fig, 'churn-drivers.svg')

top.iloc[::-1][['feature', 'coef', 'odds_ratio']].to_string(index=False)


## Qué dicen los números

- **26.5 % de baja global** (1.869 de 7.043). Es el punto de partida: sin segmentar, la media no dice nada útil.
- **El contrato mensual concentra el problema:** 42.7 % de baja frente a 11.3 % (anual) y 2.8 % (bienal), con 3.875 clientes en ese grupo. Pasar de mensual a anual es la palanca con más volumen del dataset.
- **El primer año es la ventana crítica:** 47.4 % de baja en clientes con menos de un año, y baja a 6.6 % pasados cinco años. El riesgo se juega al principio.
- **Quien se va paga más y lleva menos tiempo:** 74.44 de factura mensual media frente a 61.27 de quien se queda, y 18.0 meses de antigüedad frente a 37.6.
- **Los coeficientes coinciden con lo anterior:** contrato a dos años (odds ratio 0.26) y antigüedad (0.31) protegen; la fibra óptica (2.71) y el pago con cheque electrónico (1.44) empujan la baja.
- **AUC 0.846** con un modelo lineal simple. Frente a ese 0.846 hay que tener en cuenta el límite: el dataset no tiene fecha, así que no se puede medir *cuándo* se irá cada cliente, solo la probabilidad.

### Limitaciones

- Un solo *split* 75/25 con `random_state=42`: no se hizo validación cruzada, así que el AUC tiene variación no medida.
- El modelo es interpretable a propósito; un gradient boosting subiría el AUC y bajaría la explicabilidad. Para decidir a quién llamar, la lista ordenada por probabilidad ya sirve.
- Los datos son de una sola operadora y de un momento concreto: sirven para el método, no para extrapolar tasas a otro mercado.

### Siguiente paso

Convertir la probabilidad en dinero: cruzar el coste de la acción de retención con el valor del cliente y quedarse solo con los casos donde el ahorro esperado supera el coste. Eso es lo que convierte un modelo en una decisión.
